<!-- COMMONS LAUNCHER v4 · generated by tools/notebooks.py · do not edit by hand -->
<a href="https://github.com/TarrySingh/Artificial-Intelligence-Deep-Learning-Machine-Learning-Tutorials"><img src="https://raw.githubusercontent.com/TarrySingh/Artificial-Intelligence-Deep-Learning-Machine-Learning-Tutorials/master/brand/synapsa-commons-badge.png" alt="Synapsa Commons" height="36"></a>

Free, hands-on AI courses that run anywhere, from the team building [Synapsa](https://synapsa.realai.eu), an AI-native
learning platform.

© 2026 RealAI · free to learn from, share and adapt, not to sell ([CC BY-NC-SA 4.0](https://creativecommons.org/licenses/by-nc-sa/4.0/)).
The notice at the end of this notebook says what you may and may not do.

[![Open in Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/TarrySingh/Artificial-Intelligence-Deep-Learning-Machine-Learning-Tutorials/blob/master/lessons/T07-L02-mcp-from-the-wire-up/lesson.ipynb)
[![Open in Kaggle](https://kaggle.com/static/images/open-in-kaggle.svg)](https://kaggle.com/kernels/welcome?src=https://github.com/TarrySingh/Artificial-Intelligence-Deep-Learning-Machine-Learning-Tutorials/blob/master/lessons/T07-L02-mcp-from-the-wire-up/lesson.ipynb)
[![Open in Binder](https://mybinder.org/badge_logo.svg)](https://mybinder.org/v2/gh/TarrySingh/Artificial-Intelligence-Deep-Learning-Machine-Learning-Tutorials/master?labpath=lessons/T07-L02-mcp-from-the-wire-up/lesson.ipynb)
[![Open in Codespaces](https://github.com/codespaces/badge.svg)](https://codespaces.new/TarrySingh/Artificial-Intelligence-Deep-Learning-Machine-Learning-Tutorials)

This lesson needs Python 3.11 or newer with numpy, which Colab, Kaggle, Binder and
Codespaces already have.

In [ ]:
# --- COMMONS LAUNCHER v4 · generated by tools/notebooks.py · do not edit by hand ---
# Makes this notebook run anywhere. Every line is a no-op when the thing is already present,
# so a local clone pays nothing and an online notebook repairs itself.
import importlib.util, os, subprocess, sys, urllib.request
from pathlib import Path

COMMONS_PIP = []            # (import name, pinned pip spec) for what this lesson imports
COMMONS_SIBLINGS = []    # files that must sit beside the notebook
# A fork, a classroom mirror or an offline copy can serve the files from elsewhere by setting
# COMMONS_RAW_OVERRIDE before running this cell.
COMMONS_RAW = os.environ.get("COMMONS_RAW_OVERRIDE") or "https://raw.githubusercontent.com/TarrySingh/Artificial-Intelligence-Deep-Learning-Machine-Learning-Tutorials/master/lessons/T07-L02-mcp-from-the-wire-up/"

# Resolve siblings against the LESSON's own directory, not the working directory. A notebook
# has no __file__ and runs with cwd alongside itself; a grader imports this file from the repo
# root. Checking cwd blindly makes the grader think every sibling is missing and reach for the
# network -- which would put a download on a graded path.
try:
    COMMONS_DIR = Path(__file__).resolve().parent
except NameError:
    COMMONS_DIR = Path.cwd()


def commons_host() -> str:
    """Name the notebook service we are on. Used for the message, and for honest errors."""
    try:
        if importlib.util.find_spec("google.colab") is not None:
            return "Google Colab"
    except (ImportError, ValueError):
        pass
    if os.environ.get("KAGGLE_KERNEL_RUN_TYPE"):
        return "Kaggle"
    if os.environ.get("BINDER_SERVICE_HOST"):
        return "Binder"
    if os.environ.get("CODESPACES"):
        return "GitHub Codespaces"
    return "a local Python environment"


_missing = [pip for imp, pip in COMMONS_PIP if importlib.util.find_spec(imp) is None]
if _missing:
    print("installing " + ", ".join(_missing) + " ...")
    # pip everywhere a student is likely to be; uv-managed local venvs ship without pip.
    if importlib.util.find_spec("pip") is not None:
        subprocess.run([sys.executable, "-m", "pip", "install", "-q", *_missing], check=True)
    else:
        subprocess.run(["uv", "pip", "install", "-q", "--python", sys.executable, *_missing],
                       check=True)
    importlib.invalidate_caches()

_fetched = []
for _name in COMMONS_SIBLINGS:
    if not (COMMONS_DIR / _name).exists():
        (COMMONS_DIR / _name).parent.mkdir(parents=True, exist_ok=True)
        try:
            urllib.request.urlretrieve(COMMONS_RAW + _name, COMMONS_DIR / _name)
            _fetched.append(_name)
        except Exception as _e:  # Kaggle disables the internet by default; say so plainly
            raise RuntimeError(
                f"this lesson needs {_name} beside the notebook and could not fetch it "
                f"({_e}). On Kaggle, switch Internet on in the notebook settings panel "
                f"(Kaggle allows that only for phone-verified accounts); otherwise download it "
                f"from {COMMONS_RAW + _name} and upload it beside the notebook."
            ) from None

print("ready on " + commons_host() + ("; fetched " + ", ".join(_fetched) if _fetched else ""))
# --- END COMMONS LAUNCHER ---

# T07-L02 · The Model Context Protocol, from the wire up

**You will build:** JSON-RPC 2.0 message framing for the stdio transport, the initialize
handshake's version negotiation, a tools/list and tools/call router that tells a protocol
error apart from a tool execution error, a client that starts a real local subprocess and
walks a task to completion purely over the wire, and a conformance harness built from the
specification itself.

**Time:** ~50 minutes · **Runs on:** a laptop CPU, no GPU, no network ·
**Prerequisites:** T00-L01-the-8gb-track, T07-L01-agent-loop-from-scratch.

In [ ]:
# Setup: everything the lesson needs, in one cell, with versions printed.
import ast
import contextlib
import dis
import inspect
import io
import json
import os
import queue
import random
import subprocess
import sys
import tempfile
import textwrap
import threading
import time
import traceback
from pathlib import Path
from typing import Any, Callable, NamedTuple

import numpy as np

_LESSON_T0 = time.perf_counter()

# When section 12 relaunches this file with `--serve-mcp`, it becomes the MCP SERVER: a real
# OS subprocess speaking newline-delimited JSON-RPC on stdin/stdout. The spec is explicit that
# stdout may carry NOTHING but valid MCP messages (claims.yaml, mcp-stdio-stdout-must-not-
# carry-non-mcp-bytes), so in that mode every ordinary print() goes to stderr, and only the
# server loop in the last code cell writes to the process's real stdout, `sys.__stdout__`.
_SERVE_MODE = "--serve-mcp" in sys.argv
_REAL_STDOUT = sys.__stdout__
if _SERVE_MODE:
    sys.stdout = sys.stderr

# A notebook cell has no __file__ (there is nothing on disk to relaunch); a plain script run,
# this repository's own grader and tools/execute.py all do. Section 12's subprocess demo
# relaunches this file when it has a path, and otherwise writes a server script to launch.
try:
    _LESSON_FILE = Path(__file__).resolve()
    _HAS_FILE = True
except NameError:
    _LESSON_FILE = None
    _HAS_FILE = False

print("python", sys.version.split()[0], "· numpy", np.__version__, "· platform", sys.platform)

SEED = 20260923          # this lesson's one fixed seed: the demo chain task derives from it,
                          # so re-running prints the same chain on any machine. (Every registry
                          # below is built with transient_rate 0.0, so its own rng never fires.)

_FAILED_CHECKS: list[str] = []
_STATUS: dict[str, str] = {}   # label -> "passed" | "failed" | "not started", latest run

# The exercises, in the order you meet them, and the functions each one asks you to write.
# The progress board at the foot of the notebook is built from this, and a cell that is
# waiting on an unfinished exercise names it from here.
_EXERCISES: dict[str, tuple[str, ...]] = {
    "exercise 1": ("encode_message",),
    "exercise 2": ("decode_message",),
    "exercise 3": ("check_request_shape",),
    "exercise 4": ("negotiate_version",),
    "exercise 5": ("accept_negotiated_version",),
    "exercise 6": ("dispatch",),
    "exercise 7": ("handle_line",),
    "exercise 8": ("mcp_call",),
}


def _named(labels: list[str]) -> str:
    """["exercise 3"] -> "exercise 3 (negotiate_version)"; several -> "exercises 3, 6 and 8"."""
    if len(labels) == 1:
        return f"{labels[0]} ({', '.join(_EXERCISES[labels[0]])})"
    nums = [label.split()[-1] for label in labels]
    return "exercises " + ", ".join(nums[:-1]) + " and " + nums[-1]


def _try(label: str, check: Callable[[], None], needs: tuple[str, ...] = ()) -> None:
    """Run a check, or a demo that depends on your code, without derailing the notebook.

    A stub you have not filled in yet simply says so. A wrong answer prints the check's own
    message — which names the likely mistake — and the notebook carries on, so one broken
    exercise never hides the feedback on the others. A demo names the exercises it `needs`:
    until each has passed its check, the demo says which one it is waiting for and skips.
    Nothing is swallowed: every outcome is recorded in `_STATUS` for the progress board at the
    foot of the notebook, and every failure in `_FAILED_CHECKS`, which ends a script run
    non-zero.
    """
    waiting = [name for name in _EXERCISES
               if name in needs and _STATUS.get(name) != "passed"]
    if waiting:
        _STATUS[label] = "not started"
        print(f"{label}: skipped — needs {_named(waiting)} to pass first.")
        return
    try:
        check()
    except NotImplementedError as exc:
        _STATUS[label] = "not started"
        stub = traceback.extract_tb(exc.__traceback__)[-1].name
        owner = [name for name, funcs in _EXERCISES.items() if stub in funcs and name != label]
        if owner:
            print(f"{label}: skipped — needs {_named(owner)} first.")
        elif label in _EXERCISES:
            print(f"{label}: not implemented yet — fill in {stub}() above, then re-run "
                  "this cell.")
        else:
            print(f"{label}: skipped — {stub}() is not implemented yet.")
    except AssertionError as exc:
        _STATUS[label] = "failed"
        _FAILED_CHECKS.append(label)
        print(f"{label}: FAILED — {exc}")
    except Exception as exc:
        _STATUS[label] = "failed"
        _FAILED_CHECKS.append(label)
        print(f"{label}: raised {type(exc).__name__}: {exc}")
    else:
        _STATUS[label] = "passed"

## 1. What MCP is, and exactly which revision this lesson implements

The Model Context Protocol is JSON-RPC 2.0 messages, on an agreed transport, naming a
handful of methods (`initialize`, `tools/list`, `tools/call`, ...) — no more than that. An
SDK hands you an object with a `.call_tool()` method, and the framing, the id matching and
the error shapes all happen somewhere you never see.
This lesson implements the wire itself, over stdio, straight from the published
specification — no SDK anywhere in this file.

**Which revision, exactly.** On the date this lesson was written, the protocol's own
versioning page states plainly that its *current* revision is **2026-07-28** — and that
revision replaced the handshake below with a *different* scheme entirely: a version declared
on every request, no `initialize`, and a `server/discover` RPC every server must implement.
The 2026-07-28 versioning page names its own cut-off for the handshake-based model this
lesson teaches: **"Legacy: ... (2025-11-25 and earlier)."** This lesson implements exactly
that — revision **2025-11-25**, the most recent legacy one — because the handshake is the
part of the protocol this lesson sets out to build, and because 2026-07-28 also brings
extension negotiation and a second mandatory RPC this lesson has no room for. See
`claims.yaml` for the quotes, and section 11 for the rest of what is deliberately out of
scope.

In [ ]:
# Given: the tool registry from T07-L01, carried forward UNCHANGED (same names, same bodies —
# a minimal faithful copy, since this lesson does not re-import another lesson's files). This
# is infrastructure you already built and had graded; nothing on this page re-tests it.
class TransientToolError(Exception):
    """Raised by a tool for a failure worth retrying. Never raised for the caller's own mistake."""


class ToolResult(NamedTuple):
    ok: bool
    output: Any
    error: str | None
    transient: bool


class ToolRegistry(NamedTuple):
    tools: dict[str, Callable[..., Any]]
    schemas: dict[str, dict]


TOOL_SCHEMAS: dict[str, dict] = {
    "lookup": {"type": "object", "properties": {"key": {"type": "string"}},
              "required": ["key"]},
    "submit": {"type": "object", "properties": {"value": {"type": "integer"}},
              "required": ["value"]},
}
# New in this lesson: MCP's tools/list needs a human-readable description per tool, which
# T07-L01's registry never carried (it only needed a machine-checkable schema).
TOOL_DESCRIPTIONS: dict[str, str] = {
    "lookup": "Look up the next key (or the final integer value) for a chain key.",
    "submit": "Submit the final integer value once the chain has been walked to its end.",
}

_ARG_TYPES = {"string": str, "integer": int, "number": (int, float), "boolean": bool,
             "array": list, "object": dict}


def validate_args(schema: dict, args: dict) -> list[str]:
    """Check `args` against a documented JSON Schema subset. From T07-L01, unchanged."""
    errors: list[str] = []
    for name in schema.get("required", []):
        if name not in args:
            errors.append(f"missing required argument {name!r}")
    properties = schema.get("properties", {})
    for name, value in args.items():
        prop = properties.get(name)
        if prop is None:
            continue
        expected = prop.get("type")
        py_type = _ARG_TYPES.get(expected)
        if py_type is not None:
            reject = (isinstance(value, bool) if expected in ("integer", "number")
                      else False)
            if reject or not isinstance(value, py_type):
                errors.append(f"argument {name!r} should be of type {expected!r}, got {value!r}")
                continue
        if "enum" in prop and value not in prop["enum"]:
            errors.append(f"argument {name!r} must be one of {prop['enum']!r}, got {value!r}")
    return errors


def call_tool(registry: ToolRegistry, name: str, args: dict, validate: bool = True) -> ToolResult:
    """Execute one tool call through `registry`, never letting an exception escape. From
    T07-L01, unchanged."""
    if name not in registry.tools:
        return ToolResult(ok=False, output=None, error=f"unknown tool {name!r}", transient=False)
    schema = registry.schemas.get(name, {})
    if validate:
        errors = validate_args(schema, args)
        if errors:
            return ToolResult(ok=False, output=None,
                              error="invalid arguments: " + "; ".join(errors), transient=False)
        try:
            return ToolResult(ok=True, output=registry.tools[name](**args), error=None,
                              transient=False)
        except TransientToolError as exc:
            return ToolResult(ok=False, output=None, error=str(exc), transient=True)
        except Exception as exc:
            return ToolResult(ok=False, output=None,
                              error=f"tool {name!r} raised {type(exc).__name__}: {exc}",
                              transient=False)
    try:
        return ToolResult(ok=True, output=registry.tools[name](**args), error=None,
                          transient=False)
    except Exception as exc:
        return ToolResult(ok=False, output=None,
                          error=f"tool {name!r} raised {type(exc).__name__}: {exc}",
                          transient=True)


class ChainTask(NamedTuple):
    task_id: int
    start_key: str
    key_sequence: list[str]
    chain: dict[str, Any]
    target: int
    length: int


def generate_chain_task(length: int, seed: int) -> ChainTask:
    """One deterministic lookup chain of the given length. From T07-L01, unchanged."""
    rng = random.Random(seed)
    keys = [f"k{i}_{rng.randrange(1_000_000):06d}" for i in range(length)]
    chain: dict[str, Any] = {keys[i]: keys[i + 1] for i in range(length - 1)}
    target = rng.randint(100, 999)
    chain[keys[-1]] = target
    return ChainTask(task_id=seed, start_key=keys[0], key_sequence=keys, chain=chain,
                     target=target, length=length)


def make_registry(chain: dict, target: int, transient_rate: float,
                  rng: random.Random) -> ToolRegistry:
    """A registry over ONE task's chain. From T07-L01, unchanged."""
    def lookup(key):
        if rng.random() < transient_rate:
            raise TransientToolError("lookup timed out")
        if key not in chain:
            raise ValueError(f"unknown key {key!r}")
        val = chain[key]
        return {"next_key": val} if isinstance(val, str) else {"value": val}

    def submit(value):
        return {"correct": value == target}

    return ToolRegistry(tools={"lookup": lookup, "submit": submit}, schemas=TOOL_SCHEMAS)


# The one task this whole lesson's demos walk. Both the client role and the server role (a
# SEPARATE subprocess in section 12) build the identical registry from this same constant and
# the same fixed SEED, so the two sides agree on the chain without ever having to transmit it.
_DEMO_TASK = generate_chain_task(length=3, seed=SEED)
print(f"the demo task: start at {_DEMO_TASK.start_key!r}, length {_DEMO_TASK.length}, "
     f"target {_DEMO_TASK.target}")

## 2. New in this lesson: MCP's own vocabulary

`ServerState` is the one piece of mutable, per-connection state a dispatched request may
read or write — whether `initialize` has happened yet, and which protocol version was
negotiated. `JSONRPCParseError`, `ProtocolVersionMismatch` and `MCPClientError` are the three
ways this lesson's own code signals a problem to itself; none of them is ever the thing an
MCP *peer* sees on the wire — the wire only ever sees JSON. The four error codes below are
base JSON-RPC 2.0's own reserved codes (claims.yaml, jsonrpc-2-0-reserved-error-codes);
exercises 6 and 7 answer every protocol error with one of them. Given to you.

In [ ]:
class ServerState:
    """Mutable per-connection state. `registry` is fixed for the connection's life."""

    def __init__(self, registry: ToolRegistry):
        self.registry = registry
        self.initialized = False
        self.negotiated_version: str | None = None


class JSONRPCParseError(Exception):
    """Raised by decode_message when a line is not valid JSON."""


class ProtocolVersionMismatch(Exception):
    """Raised by accept_negotiated_version when the server named a version this client does
    not itself support."""


class MCPClientError(Exception):
    """Raised by mcp_call when a response's id does not match the request that was sent."""


def _protocol_error(request_id: Any, code: int, message: str) -> dict:
    """Build a JSON-RPC error response object. Given to you — every exercise below builds on
    this rather than re-typing the {"jsonrpc": "2.0", ...} envelope by hand."""
    return {"jsonrpc": "2.0", "id": request_id, "error": {"code": code, "message": message}}


PARSE_ERROR = -32700        # the line is not valid JSON at all
INVALID_REQUEST = -32600    # valid JSON, but not a valid JSON-RPC request object
METHOD_NOT_FOUND = -32601   # a well-formed request naming a method this server does not have
INVALID_PARAMS = -32602     # the method exists, but its params are wrong (MCP: unknown tool too)

SUPPORTED_VERSIONS = ("2025-06-18", "2025-11-25")   # the two revisions this server/client know
LATEST_VERSION = max(SUPPORTED_VERSIONS)             # ISO dates sort chronologically as strings
PROTOCOL_VERSION = LATEST_VERSION                    # this side's OWN preferred version to send
SERVER_INFO = {"name": "t07-l02-chain-server", "version": "1.0.0"}
CLIENT_INFO = {"name": "t07-l02-chain-client", "version": "1.0.0"}

print(f"this lesson speaks MCP revision {PROTOCOL_VERSION} (also accepts "
     f"{[v for v in SUPPORTED_VERSIONS if v != LATEST_VERSION]})")

## 3. Exercise 1 — `encode_message()`

Over stdio, MCP is JSON-RPC: UTF-8 bytes, one message per line, and a message must never
contain an embedded newline (see claims.yaml). `json.dumps` already writes a newline *inside
a string value* as the two characters `\` and `n`, so the only way a real newline byte gets
into the line is formatting you ask for: `json.dumps(..., indent=2)` puts one between every
key, and breaks the framing. Leave `indent` alone and a message is one line by construction.

<details><summary>💡 Hint 1 — what to think about</summary>

One line of JSON, terminated by exactly one newline, encoded as UTF-8 bytes. The docstring's
example has no space after any comma or colon: which `json.dumps` keyword argument controls
the text written between items and between a key and its value?
</details>
<details><summary>💡 Hint 2 — the approach, in words</summary>

Serialise the dict to a string with that keyword set to a comma and a colon and nothing
else, add the single newline terminator to the end of the string, and turn the whole string
into bytes with the UTF-8 codec in one step at the very end.
</details>

In [ ]:
def encode_message(msg: dict) -> bytes:
    """Frame one JSON-RPC message for the stdio wire: compact JSON, UTF-8, one trailing `\\n`.

    Returns bytes ending in exactly one `\\n`, with no other newline anywhere in the line —
    which any encoding without `indent` gives you for free, since json.dumps writes a newline
    inside a string value as the two characters `\\` and `n`, never literally.

    Example:
        >>> encode_message({"jsonrpc": "2.0", "id": 1, "method": "ping"})
        b'{"jsonrpc":"2.0","id":1,"method":"ping"}\\n'
    """
    # YOUR CODE HERE
    raise NotImplementedError


def _check_encode_message() -> None:
    out = encode_message({"jsonrpc": "2.0", "id": 1, "method": "ping"})
    assert isinstance(out, bytes), f"encode_message must return bytes, got {type(out)}"
    assert out.endswith(b"\n"), f"got {out!r} — every message must end with exactly one newline"
    assert out.count(b"\n") == 1, (
        f"got {out!r} with {out.count(chr(10).encode())} newline(s) — a pretty-printed "
        "(multi-line) encoding breaks stdio's one-message-per-line framing; use "
        'separators=(",", ":") for a compact, single-line encoding'
    )
    decoded = json.loads(out.decode("utf-8").rstrip("\n"))
    assert decoded == {"jsonrpc": "2.0", "id": 1, "method": "ping"}, (
        f"round-tripping {out!r} through json.loads did not return the original message"
    )
    with_newline_in_value = encode_message({"jsonrpc": "2.0", "id": 2, "method": "x",
                                            "params": {"note": "line one\nline two"}})
    assert with_newline_in_value.count(b"\n") == 1, (
        "a value that itself contains a newline character must still produce exactly ONE "
        "newline byte in the encoded line — json.dumps escapes it as the two characters "
        r"\ and n inside the string; only the trailing terminator is a real newline byte"
    )
    unicode_line = encode_message({"jsonrpc": "2.0", "id": 3, "method": "x", "params": {"s": "café"}})
    unicode_line.decode("utf-8")   # must not raise: the bytes must be valid UTF-8 either way
    assert json.loads(unicode_line.decode("utf-8"))["params"]["s"] == "café", (
        "a non-ASCII character must round-trip through encode_message unchanged, however "
        "json.dumps chooses to represent it on the wire (escaped \\uXXXX or raw UTF-8 are "
        "both valid JSON)"
    )


_try("exercise 1", _check_encode_message)

## 4. Exercise 2 — `decode_message()`

The other half of framing: turn one line of bytes back into the JSON value it encodes. A
line that is not valid JSON at all — truncated, or plain garbage — is not this lesson's
problem to solve gracefully by guessing; it is exercise 7's problem, which needs to know
*that* parsing failed so it can build the right JSON-RPC error. This function's job is only
to say so, through a typed exception, never a bare `json.JSONDecodeError` a caller has to
know to expect.

<details><summary>💡 Hint 1 — what to think about</summary>

Decode the bytes as UTF-8 first (a stray non-UTF-8 byte is also a framing problem, not
silently ignorable), strip the trailing line terminator, then `json.loads`. Which exception
does `json.loads` raise on invalid JSON, and what do you need to catch it and re-raise as?
</details>
<details><summary>💡 Hint 2 — the approach, in words</summary>

Two things can go wrong, and each has its own exception: turning the bytes into text, and
turning the text into a value. Guard both, and in each handler raise the lesson's own parse
error instead, chained to the original so its traceback survives for anyone debugging.
Strip any trailing carriage return or newline from the text first: a line read from a real
stream still has its terminator attached.
</details>

In [ ]:
def decode_message(line: bytes) -> Any:
    """Parse one framed line back into the JSON value it encodes.

    Accepts the line with or without its trailing newline. Raises `JSONRPCParseError` — never
    a bare `json.JSONDecodeError` or `UnicodeDecodeError` — if the bytes are not valid UTF-8
    JSON.

    Example:
        >>> decode_message(b'{"jsonrpc":"2.0","id":1,"method":"ping"}\\n')
        {'jsonrpc': '2.0', 'id': 1, 'method': 'ping'}
        >>> decode_message(b'not json')
        Traceback (most recent call last):
            ...
        JSONRPCParseError: ...
    """
    # YOUR CODE HERE
    raise NotImplementedError


def _check_decode_message() -> None:
    msg = decode_message(b'{"jsonrpc":"2.0","id":1,"method":"ping"}\n')
    assert msg == {"jsonrpc": "2.0", "id": 1, "method": "ping"}, f"got {msg!r}"
    no_newline = decode_message(b'{"jsonrpc":"2.0","id":1,"method":"ping"}')
    assert no_newline == msg, "a line with no trailing newline must decode identically to one with"
    try:
        decode_message(b"not json at all")
    except JSONRPCParseError:
        pass
    else:
        raise AssertionError(
            "decode_message(b'not json at all') did not raise — invalid JSON must raise "
            "JSONRPCParseError, not return None or propagate json.JSONDecodeError directly"
        )
    try:
        decode_message(b'{"unterminated": ')
    except JSONRPCParseError:
        pass
    else:
        raise AssertionError("truncated JSON must also raise JSONRPCParseError")
    try:
        decode_message(b"\xff\xfe not utf-8")
    except JSONRPCParseError:
        pass
    else:
        raise AssertionError(
            "bytes that are not valid UTF-8 at all must raise JSONRPCParseError too, not an "
            "uncaught UnicodeDecodeError"
        )
    round_trip = decode_message(encode_message({"jsonrpc": "2.0", "id": "abc", "result": {}}))
    assert round_trip == {"jsonrpc": "2.0", "id": "abc", "result": {}}, (
        "decode_message(encode_message(x)) must return x unchanged"
    )


_try("exercise 2", _check_decode_message)

## 5. Exercise 3 — `check_request_shape()`

Before anything asks *what* a message wants, check that it is even a well-formed JSON-RPC
message MCP will accept. Two rules matter enough to be graded on their own: `params`, if
present, must be a JSON *object* (MCP never uses JSON-RPC's positional array form), and —
stricter than base JSON-RPC — an `id` that is explicitly `null` is INVALID here, not a quiet
way of marking a notification (see claims.yaml). A notification is a message with the `id`
key *absent entirely*; `"id": null` is a different, malformed thing.

<details><summary>💡 Hint 1 — what to think about</summary>

Five independent things to check, each returning immediately with its own message: is `msg`
even a dict; is `"jsonrpc"` exactly the string `"2.0"`; is `"method"` a non-empty string; if
`"id"` is present, is it a non-null `str` or `int` (and not a `bool` — the same JSON-has-no-
such-rule trap as T07-L01's `validate_args`); if `"params"` is present, is it a dict.
</details>
<details><summary>💡 Hint 2 — the approach, in words</summary>

Decide whether to look at the id by asking whether the KEY is in the dict, not by reading
its value: a missing key and a null value read back the same way, and that is exactly the
distinction this exercise is about. Once you know the key is there, a null value is invalid,
and so is anything that is not a string or an integer. Rule out booleans before the integer
test, because Python counts a bool as an int.
</details>

In [ ]:
def check_request_shape(msg: Any) -> str | None:
    """Is `msg` a JSON-RPC message MCP will accept? Returns an error string naming the first
    problem found, or None if it is valid — as either a request (has "id") or a notification
    (does not).

    Example:
        >>> check_request_shape({"jsonrpc": "2.0", "id": 1, "method": "ping"}) is None
        True
        >>> check_request_shape({"jsonrpc": "2.0", "id": None, "method": "ping"})
        "'id' must not be null"
    """
    # YOUR CODE HERE
    raise NotImplementedError


def _check_check_request_shape() -> None:
    ok_request = check_request_shape({"jsonrpc": "2.0", "id": 1, "method": "tools/list"})
    assert ok_request is None, f"a valid request must return None, got {ok_request!r}"
    ok_notification = check_request_shape({"jsonrpc": "2.0", "method": "notifications/initialized"})
    assert ok_notification is None, (
        f"a valid notification (no 'id' key at all) must return None, got {ok_notification!r}"
    )
    not_dict = check_request_shape(["jsonrpc", "2.0"])
    assert not_dict is not None, "a JSON array at the top level is not a valid message"
    wrong_version = check_request_shape({"jsonrpc": "1.0", "id": 1, "method": "x"})
    assert wrong_version is not None, "jsonrpc must be exactly \"2.0\""
    no_method = check_request_shape({"jsonrpc": "2.0", "id": 1})
    assert no_method is not None, "a message with no 'method' at all must be rejected"
    empty_method = check_request_shape({"jsonrpc": "2.0", "id": 1, "method": ""})
    assert empty_method is not None, "an empty string method name must be rejected"
    number_method = check_request_shape({"jsonrpc": "2.0", "id": 1, "method": 5})
    assert number_method is not None, (
        "a method that is not a string at all must be rejected — testing only that the method "
        "is truthy lets a number through"
    )
    zero_id = check_request_shape({"jsonrpc": "2.0", "id": 0, "method": "x"})
    assert zero_id is None, (
        f"got {zero_id!r} for id 0 — zero is a valid integer id; the null test must ask "
        "'is it None?', not 'is it falsy?'"
    )
    null_id = check_request_shape({"jsonrpc": "2.0", "id": None, "method": "x"})
    assert null_id is not None and "null" in null_id, (
        f"got {null_id!r} — MCP forbids id: null explicitly (unlike base JSON-RPC); this must "
        "be flagged, and the message must NOT be treated as a silent notification"
    )
    bool_id = check_request_shape({"jsonrpc": "2.0", "id": True, "method": "x"})
    assert bool_id is not None, (
        "id: true must be rejected — isinstance(True, int) is True in Python, but a JSON "
        "boolean is not a JSON-RPC id"
    )
    float_id = check_request_shape({"jsonrpc": "2.0", "id": 1.5, "method": "x"})
    assert float_id is not None, "a non-integer number is not a valid id either"
    string_id = check_request_shape({"jsonrpc": "2.0", "id": "req-1", "method": "x"})
    assert string_id is None, "a string id is explicitly allowed and must pass"
    array_params = check_request_shape({"jsonrpc": "2.0", "id": 1, "method": "x", "params": [1, 2]})
    assert array_params is not None, (
        "MCP's params must be a JSON object — positional array-style params (legal in base "
        "JSON-RPC) must still be rejected here"
    )
    object_params = check_request_shape({"jsonrpc": "2.0", "id": 1, "method": "x", "params": {}})
    assert object_params is None, "an empty object is a perfectly valid params value"


_try("exercise 3", _check_check_request_shape)

## 6. Exercise 4 — `negotiate_version()`

The rule `initialize` uses to agree a protocol version (see claims.yaml): if the server
supports the version the client asked for, it MUST echo that SAME version back. Otherwise it
MUST still answer with a version it *does* support — and SHOULD make that its latest —
inside an ordinary successful result, so the client can decide for itself whether it can
speak that version (exercise 5). The same lifecycle page also shows an example *error*
response for an unsupported version and tells implementations to be prepared for one
(claims.yaml, mcp-lifecycle-unsupported-version-error-example) — a robust client handles
both shapes — but a server that follows the MUST rule answers with a version, and that is
the server you are building. Answering with an error, or echoing the
unsupported version back, is the most likely mistake in this lesson.

<details><summary>💡 Hint 1 — what to think about</summary>

Two branches only: `requested` is exactly one of the versions this side supports, or it is
not. In the second branch, which single version from `supported` should come back, and what
built-in, given that ISO `YYYY-MM-DD` strings sort exactly like the dates they name, finds it
without hand-picking an index?
</details>
<details><summary>💡 Hint 2 — the approach, in words</summary>

A membership test decides the branch. On a hit, hand the request straight back. On a miss,
return the greatest string in the supported tuple: dates written year-month-day compare as
plain strings in date order, so the greatest is the latest, whatever order the tuple is in
and whatever the client asked for. Nothing in either branch raises.
</details>

In [ ]:
def negotiate_version(requested: str, supported: tuple[str, ...]) -> str:
    """The version `initialize` should answer with, given what the client asked for.

    If `requested` is one of `supported`, it is echoed back unchanged. Otherwise the LATEST
    version in `supported` is returned instead. Never raises: an unsupported request is
    answered with a version, not an error (see claims.yaml,
    mcp-initialize-version-negotiation-rule).

    Example:
        >>> negotiate_version("2025-06-18", ("2025-06-18", "2025-11-25"))
        '2025-06-18'
        >>> negotiate_version("1999-01-01", ("2025-06-18", "2025-11-25"))
        '2025-11-25'
    """
    # YOUR CODE HERE
    raise NotImplementedError


def _check_negotiate_version() -> None:
    latest = negotiate_version(LATEST_VERSION, SUPPORTED_VERSIONS)
    assert latest == LATEST_VERSION, f"requesting the latest supported version must echo it back"
    other_supported = [v for v in SUPPORTED_VERSIONS if v != LATEST_VERSION][0]
    echoed = negotiate_version(other_supported, SUPPORTED_VERSIONS)
    assert echoed == other_supported, (
        f"got {echoed!r} — requesting a version that IS supported (just not the latest) must "
        f"be echoed back UNCHANGED, not silently upgraded to {LATEST_VERSION!r}"
    )
    unsupported = negotiate_version("1999-01-01", SUPPORTED_VERSIONS)
    assert unsupported == LATEST_VERSION, (
        f"got {unsupported!r} for a version this side does not support — the fallback must be "
        f"this side's OWN latest supported version ({LATEST_VERSION!r}), never the requested "
        "value echoed back and never an exception"
    )
    garbage = negotiate_version("not-a-version-at-all", SUPPORTED_VERSIONS)
    assert garbage == LATEST_VERSION, "a nonsense string must fall back the same way"
    between = negotiate_version("2025-09-01", SUPPORTED_VERSIONS)
    assert between == LATEST_VERSION, (
        f"got {between!r} for a date that falls BETWEEN two supported versions — the fallback "
        "is always this side's latest, not the nearest supported version below the request"
    )
    two_versions = ("2024-01-01", "2026-01-01")
    picked = negotiate_version("2020-01-01", two_versions)
    assert picked == "2026-01-01", (
        f"got {picked!r} — the LATEST of the two supported versions must be chosen, not the "
        "first one in the tuple or the smallest"
    )
    reversed_pick = negotiate_version("2020-01-01", ("2026-01-01", "2024-01-01"))
    assert reversed_pick == "2026-01-01", (
        f"got {reversed_pick!r} with the tuple in the other order — the latest must be found by "
        "comparing the dates, not by its position in the tuple"
    )


_try("exercise 4", _check_negotiate_version)

## 7. Exercise 5 — `accept_negotiated_version()`

The client's half of negotiation. Once the server has answered `initialize`, the client
checks the version the server actually chose: if it is one the client itself can speak, all
is well; if not, the client "SHOULD disconnect" (claims.yaml). This lesson models
"disconnect" as an exception that stops the caller (section 12's demo) before it sends
another request, rather than silently carrying on with a version it cannot actually speak.

<details><summary>💡 Hint 1 — what to think about</summary>

One membership test. What should happen on each of the two outcomes — and which one needs a
message that actually names the mismatched version, for whoever reads it later?
</details>
<details><summary>💡 Hint 2 — the approach, in words</summary>

If the server's version is not among the ones this client supports, raise the lesson's
version-mismatch exception with a message that names the server's version. On the accepting
path there is nothing to do and nothing to return.
</details>

In [ ]:
def accept_negotiated_version(client_supported: frozenset[str], server_version: str) -> None:
    """Accept the server's negotiated version, or raise if this client cannot speak it.

    Example:
        >>> accept_negotiated_version(frozenset({"2025-11-25"}), "2025-11-25")  # no error
        >>> accept_negotiated_version(frozenset({"2025-11-25"}), "2019-01-01")
        Traceback (most recent call last):
            ...
        ProtocolVersionMismatch: ...
    """
    # YOUR CODE HERE
    raise NotImplementedError


def _check_accept_negotiated_version() -> None:
    accept_negotiated_version(frozenset(SUPPORTED_VERSIONS), LATEST_VERSION)  # must not raise
    try:
        accept_negotiated_version(frozenset({"2025-11-25"}), "2019-01-01")
    except ProtocolVersionMismatch as exc:
        assert "2019-01-01" in str(exc), (
            f"the raised message should name the mismatched version, got: {exc}"
        )
    else:
        raise AssertionError(
            "a server_version outside client_supported must raise ProtocolVersionMismatch — "
            "silently accepting it means later calls might speak a version this client cannot"
        )
    try:
        accept_negotiated_version(frozenset(), "2025-11-25")
    except ProtocolVersionMismatch:
        pass
    else:
        raise AssertionError("an empty client_supported set can never accept any version")


_try("exercise 5", _check_accept_negotiated_version)

## 8. Exercise 6 — `dispatch()`

The router: given already-validated state and a parsed message, decide what happens. This
server implements `initialize`, `ping`, `tools/list` and `tools/call` (section 11 says why
nothing else). Whether a response is sent at all is decided by structure, not by the method
name: a message with no `"id"` key is a notification, and a notification never gets a
response (claims.yaml, mcp-notification-must-not-include-id). Testing
`method.startswith("notifications/")` instead would silently drop a genuine request that
happens to share that prefix, and answer a notification sent under any other name.

`tools/call`'s error handling is the graded heart of this exercise: an unknown tool name, or
a call missing its own required `"name"`, is a PROTOCOL error (a JSON-RPC `error`,
`INVALID_PARAMS`); a known tool whose ARGUMENTS fail ITS OWN schema — T07-L01's
`validate_args`, reached through `call_tool` — is a TOOL EXECUTION error, reported inside a
SUCCESSFUL result with `"isError": true` (claims.yaml, mcp-tools-error-two-mechanisms). A
client that cannot tell them apart cannot tell "that tool does not exist" from "that tool
says your input was wrong, and why", and only the second is worth handing back to a model.

<details><summary>💡 Hint 1 — what to think about</summary>

Look for the `"id"` key FIRST, before looking at `method` at all, and return `None` from that
branch whatever the method is. For `tools/call`: is `params` even shaped right (a dict with
a string `"name"`, and an `"arguments"` object if present) before you look anything up? Is
`name` actually IN the registry before you call it? Only once those pass does `call_tool`'s
own `ok` flag choose between the two result shapes in the docstring.
</details>
<details><summary>💡 Hint 2 — the approach, in words</summary>

After the notification test, branch on the method name. Initialize: negotiate with exercise
4, record the outcome on `state`, and build the result the docstring describes. Ping: an
empty result. Tools/list: one entry per schema in the registry, each with its description.
Tools/call: reject a badly shaped params or an unknown name with the given error helper and
the invalid-params code; otherwise run the tool through `call_tool` and report its outcome as
a result either way — never as a JSON-RPC error. Any other method: the method-not-found code.
</details>

In [ ]:
def dispatch(state: ServerState, msg: dict) -> dict | None:
    """Route one already-shape-checked message to a response, or None for a notification.

    `msg` has already passed `check_request_shape`. A message with no "id" key is a
    notification and this function ALWAYS returns None for it, whatever "method" says. Every
    other message gets {"jsonrpc": "2.0", "id": <the request's id>, "result": ...} or an error
    built with `_protocol_error`. The results, by method:

      "initialize"  negotiate params["protocolVersion"] with `negotiate_version`, set
                    state.initialized and state.negotiated_version, and return
                    {"protocolVersion": <negotiated>, "capabilities": {"tools": {}},
                     "serverInfo": SERVER_INFO}
      "ping"        {}  (an empty result)
      "tools/list"  {"tools": [{"name": ..., "description": TOOL_DESCRIPTIONS[name],
                                "inputSchema": <its schema>}, ...]}, one per registry schema
      "tools/call"  params {"name": str, "arguments": dict (optional, default {})}: a bad
                    shape or an unknown name is an INVALID_PARAMS error; otherwise
                    `call_tool(state.registry, name, arguments)` becomes
                    {"content": [{"type": "text", "text": <text>}], "isError": <not ok>},
                    where <text> is json.dumps(output) on success and the error string on
                    failure
      anything else a METHOD_NOT_FOUND error whose message names the method

    Example:
        >>> reg = make_registry(_DEMO_TASK.chain, _DEMO_TASK.target, 0.0, random.Random(0))
        >>> state = ServerState(registry=reg)
        >>> dispatch(state, {"jsonrpc": "2.0", "method": "notifications/initialized"}) is None
        True
        >>> r = dispatch(state, {"jsonrpc": "2.0", "id": 1, "method": "tools/list"})
        >>> sorted(t["name"] for t in r["result"]["tools"])
        ['lookup', 'submit']
    """
    # YOUR CODE HERE
    raise NotImplementedError


def _check_dispatch() -> None:
    def fresh_state() -> ServerState:
        reg = make_registry(_DEMO_TASK.chain, _DEMO_TASK.target, 0.0, random.Random(0))
        return ServerState(registry=reg)

    notif = dispatch(fresh_state(), {"jsonrpc": "2.0", "method": "notifications/initialized"})
    assert notif is None, "a message with no 'id' must return None, regardless of its method"
    odd_notif = dispatch(fresh_state(), {"jsonrpc": "2.0", "method": "tools/list"})
    assert odd_notif is None, (
        "a notification (no 'id') naming a METHOD THAT IS NOT notifications/... must STILL get "
        "no response — the rule is the presence of 'id', never the method name string"
    )

    state = fresh_state()
    init = dispatch(state, {"jsonrpc": "2.0", "id": 1, "method": "initialize",
                            "params": {"protocolVersion": LATEST_VERSION}})
    assert init is not None and "result" in init, (
        f"got {init!r} — initialize with a supported version must return a successful result"
    )
    assert init["result"].get("protocolVersion") == LATEST_VERSION and init["id"] == 1, (
        f"got {init!r} — a supported version must be echoed back, under the request's own id"
    )
    assert "tools" in init["result"].get("capabilities", {}), (
        f"got {init['result']!r} — a server that offers tools must declare a 'tools' entry in "
        "its capabilities (claims.yaml, mcp-tools-capability-must-be-declared)"
    )
    assert init["result"].get("serverInfo") == SERVER_INFO, (
        f"got {init['result']!r} — the result must carry serverInfo: SERVER_INFO"
    )
    assert state.initialized is True and state.negotiated_version == LATEST_VERSION, (
        "initialize must record state.initialized = True and the negotiated version on state"
    )

    old_version = [v for v in SUPPORTED_VERSIONS if v != LATEST_VERSION][0]
    init_old = dispatch(fresh_state(), {"jsonrpc": "2.0", "id": 2, "method": "initialize",
                                        "params": {"protocolVersion": old_version}})
    assert init_old.get("result", {}).get("protocolVersion") == old_version, (
        f"got {init_old!r} — requesting a SUPPORTED (non-latest) version must echo it back"
    )

    init_bad = dispatch(fresh_state(), {"jsonrpc": "2.0", "id": 3, "method": "initialize",
                                        "params": {"protocolVersion": "1900-01-01"}})
    assert "error" not in init_bad, (
        f"got {init_bad!r} — under the MUST rule a server answers an UNSUPPORTED "
        "protocolVersion with a successful result naming a version it DOES support, not an error"
    )
    assert init_bad["result"].get("protocolVersion") == LATEST_VERSION, (
        f"got {init_bad!r} — the fallback must be this server's own latest supported version"
    )

    ping = dispatch(fresh_state(), {"jsonrpc": "2.0", "id": "p1", "method": "ping"})
    assert ping == {"jsonrpc": "2.0", "id": "p1", "result": {}}, (
        f"got {ping!r} — ping must be answered with an EMPTY result under the request's id "
        "(claims.yaml, mcp-ping-must-respond-empty), not with Method not found"
    )

    listed = dispatch(fresh_state(), {"jsonrpc": "2.0", "id": 4, "method": "tools/list"})
    names = {t["name"] for t in listed["result"]["tools"]}
    assert names == {"lookup", "submit"}, f"got {names}"
    assert all(t["inputSchema"] == TOOL_SCHEMAS[t["name"]] for t in listed["result"]["tools"]), (
        "each tool's inputSchema must be exactly its entry in TOOL_SCHEMAS"
    )

    unknown = dispatch(fresh_state(), {"jsonrpc": "2.0", "id": 5, "method": "tools/call",
                                       "params": {"name": "not_a_real_tool", "arguments": {}}})
    assert unknown.get("error", {}).get("code") == INVALID_PARAMS, (
        f"an unknown tool must be a PROTOCOL error (INVALID_PARAMS), got {unknown!r} — call_tool "
        "would report it as an execution error, so check the registry BEFORE calling it"
    )
    assert "not_a_real_tool" in unknown["error"]["message"], (
        "the error message should name the unknown tool, matching the spec's own "
        "'Unknown tool: <name>' example (claims.yaml, mcp-tools-unknown-tool-example)"
    )

    missing_name = dispatch(fresh_state(), {"jsonrpc": "2.0", "id": 6, "method": "tools/call",
                                            "params": {"arguments": {"key": "x"}}})
    assert missing_name.get("error", {}).get("code") == INVALID_PARAMS, (
        f"a tools/call missing its required 'name' must be a protocol error too, got {missing_name!r}"
    )

    bad_args_shape = dispatch(fresh_state(), {"jsonrpc": "2.0", "id": 7, "method": "tools/call",
                                              "params": {"name": "lookup", "arguments": "not-an-object"}})
    assert bad_args_shape.get("error", {}).get("code") == INVALID_PARAMS, (
        f"an 'arguments' value that is not a JSON object must be a PROTOCOL error, got "
        f"{bad_args_shape!r} — this is a malformed tools/call REQUEST, not a bad TOOL argument"
    )

    bad_args = dispatch(fresh_state(), {"jsonrpc": "2.0", "id": 7, "method": "tools/call",
                                        "params": {"name": "lookup", "arguments": {}}})
    assert "error" not in bad_args, (
        f"a KNOWN tool called with arguments that fail ITS OWN schema must NOT be a JSON-RPC "
        f"error — got {bad_args!r}; it must be a successful result with isError: true"
    )
    assert bad_args["result"].get("isError") is True, (
        f"got {bad_args['result']!r} — a validate_args failure is a TOOL EXECUTION error "
        "(isError: true inside a successful result), not a protocol error"
    )
    content = bad_args["result"].get("content") or [{}]
    assert content[0].get("type") == "text" and "key" in content[0].get("text", ""), (
        f"got {bad_args['result']!r} — an execution error must still say WHAT went wrong, as "
        "a text content item carrying call_tool's error string, so a model can fix its input"
    )

    good_call = dispatch(fresh_state(), {"jsonrpc": "2.0", "id": 8, "method": "tools/call",
                                         "params": {"name": "lookup",
                                                    "arguments": {"key": _DEMO_TASK.start_key}}})
    assert good_call.get("result", {}).get("isError") is False, f"got {good_call!r}"
    payload = json.loads(good_call["result"]["content"][0]["text"])
    assert payload == {"next_key": _DEMO_TASK.key_sequence[1]}, (
        f"got {payload!r} — the text content of a successful call is json.dumps of the tool's "
        "own output"
    )

    unknown_method = dispatch(fresh_state(), {"jsonrpc": "2.0", "id": 9, "method": "resources/list"})
    assert unknown_method.get("error", {}).get("code") == METHOD_NOT_FOUND, (
        f"a method this server does not implement at all must be METHOD_NOT_FOUND, "
        f"got {unknown_method!r}"
    )


_try("exercise 6", _check_dispatch)

## 9. Exercise 7 — `handle_line()`

One line in, at most one line out: this is the function section 12's real subprocess server
loop calls for every line it reads. It is the composition of everything above — decode, then
shape-check, then dispatch — and it is where an unreadable id gets handled. Base JSON-RPC
says that when a line cannot even be parsed, or is too malformed to trust its `id`, the
error response's `id` MUST be `null` (claims.yaml, jsonrpc-2-0-error-id-must-be-null). MCP
2025-11-25's own schema instead makes that `id` optional in exactly this case (claims.yaml,
mcp-error-response-id-when-unreadable). This server sends `null`, which satisfies the JSON-RPC
rule MCP says every message must follow; a peer that omits the key is following MCP's schema,
and every check here accepts either. What neither allows is guessing an id.

<details><summary>💡 Hint 1 — what to think about</summary>

Three steps in order, each able to end the function early: parsing can fail (id unknown,
so null); shape-checking can fail (the id IF one can safely be read, else null); otherwise
`dispatch` decides, and its answer (a dict, or `None`) is what you return, encoded or not.
</details>
<details><summary>💡 Hint 2 — the approach, in words</summary>

Catch the lesson's parse error around exercise 2 and answer it with an encoded error built
from the given helper and the parse-error code. If exercise 3 names a problem, answer with
the invalid-request code, echoing the message's own id only when the message is a dict and
that id passes the same string-or-integer, not-a-bool test as exercise 3; otherwise null.
Everything else goes to `dispatch`: encode what it returns, or pass its `None` straight on.
</details>

In [ ]:
def handle_line(state: ServerState, raw: bytes) -> bytes | None:
    """Handle one raw line from the wire: decode, shape-check, dispatch, encode.

    Returns the encoded response line to write back -- one JSON message ending in exactly one
    newline, as encode_message produces -- or None if nothing should be written at all (the
    message was a valid notification). A line that fails to parse is answered with a
    PARSE_ERROR; one that parses but fails `check_request_shape` with an INVALID_REQUEST. Both
    carry "id": null when no valid id can be read (claims.yaml,
    jsonrpc-2-0-error-id-must-be-null), and the request's own id when one can.

    Example:
        >>> reg = make_registry(_DEMO_TASK.chain, _DEMO_TASK.target, 0.0, random.Random(0))
        >>> state = ServerState(registry=reg)
        >>> out = handle_line(state, b"not json\\n")
        >>> json.loads(out)["error"]["code"]
        -32700
    """
    # YOUR CODE HERE
    raise NotImplementedError


def _check_handle_line() -> None:
    def fresh_state() -> ServerState:
        reg = make_registry(_DEMO_TASK.chain, _DEMO_TASK.target, 0.0, random.Random(0))
        return ServerState(registry=reg)

    malformed = handle_line(fresh_state(), b'{"jsonrpc": "2.0", "id": 1, "method": ')
    assert malformed is not None, "malformed JSON must still get an answer, not silence"
    assert malformed.endswith(b"\n") and malformed.count(b"\n") == 1, (
        f"got {malformed!r} — every reply must be ONE framed line ending in exactly one newline "
        "(send it through encode_message); without the newline the client waits forever"
    )
    parsed = json.loads(malformed)
    assert parsed.get("id") is None and parsed.get("error", {}).get("code") == PARSE_ERROR, (
        f"got {parsed!r} — a line that will not parse as JSON must come back as a JSON-RPC "
        "error with PARSE_ERROR and a null id (no id can be trusted if the JSON never parsed)"
    )

    unknown_method = handle_line(fresh_state(), encode_message(
        {"jsonrpc": "2.0", "id": 7, "method": "tools/frobnicate"}))
    parsed = json.loads(unknown_method)
    assert parsed.get("id") == 7 and parsed.get("error", {}).get("code") == METHOD_NOT_FOUND, (
        f"got {parsed!r} — the request's OWN id must come back on a routing error, unlike a "
        "parse error, since here the id WAS readable"
    )

    shape_invalid_but_id_readable = handle_line(fresh_state(), encode_message(
        {"jsonrpc": "1.0", "id": 42, "method": "tools/list"}))
    parsed = json.loads(shape_invalid_but_id_readable)
    assert parsed.get("id") == 42 and parsed.get("error", {}).get("code") == INVALID_REQUEST, (
        f"got {parsed!r} — when the id CAN be safely read (a valid str/int, here 42), an "
        "Invalid Request error must still echo it back, unlike a genuine parse error where "
        "no id can be trusted at all"
    )

    bool_id = handle_line(fresh_state(), encode_message(
        {"jsonrpc": "2.0", "id": True, "method": "tools/list"}))
    parsed = json.loads(bool_id)
    assert parsed.get("error", {}).get("code") == INVALID_REQUEST and parsed.get("id") is None, (
        f"got {parsed!r} — id: true is not a valid id, so the Invalid Request answer must not "
        "echo it: only a str or int (never a bool) id may be copied into the response"
    )

    null_id = handle_line(fresh_state(), encode_message(
        {"jsonrpc": "2.0", "id": None, "method": "tools/list"}))
    assert null_id is not None, "id: null is an INVALID request, not a silent notification"
    parsed = json.loads(null_id)
    assert parsed.get("error", {}).get("code") == INVALID_REQUEST and parsed.get("id") is None, (
        f"got {parsed!r} — id: null must be answered with INVALID_REQUEST, and the answer "
        "cannot carry an id the request never validly had"
    )

    notif = handle_line(fresh_state(), encode_message(
        {"jsonrpc": "2.0", "method": "notifications/initialized"}))
    assert notif is None, (
        f"got {notif!r} for a valid notification — handle_line must return None, meaning "
        "write NOTHING back, not an empty bytes object or a JSON null"
    )

    wrong_version = handle_line(fresh_state(), encode_message(
        {"jsonrpc": "2.0", "id": 1, "method": "initialize",
         "params": {"protocolVersion": "1999-01-01"}}))
    parsed = json.loads(wrong_version)
    assert "error" not in parsed and parsed["result"]["protocolVersion"] == LATEST_VERSION, (
        f"got {parsed!r} — an unsupported protocolVersion routed through the full pipeline "
        "must still be a successful result, exactly as exercise 6 grades on its own"
    )

    missing_params = handle_line(fresh_state(), encode_message(
        {"jsonrpc": "2.0", "id": 2, "method": "tools/call"}))
    parsed = json.loads(missing_params)
    assert parsed.get("error", {}).get("code") == INVALID_PARAMS, (
        f"a tools/call with NO params key at all must still be INVALID_PARAMS, got {parsed!r}"
    )

    good = handle_line(fresh_state(), encode_message(
        {"jsonrpc": "2.0", "id": 3, "method": "tools/list"}))
    assert isinstance(good, bytes), f"handle_line must return encoded bytes, got {type(good)}"
    parsed = json.loads(good)
    assert {t["name"] for t in parsed["result"]["tools"]} == {"lookup", "submit"}, f"got {parsed!r}"


_try("exercise 7", _check_handle_line)

**A conformance harness.** Exercise 7's check tested `handle_line` one assertion at a time.
The harness below turns the same wire rules into DATA — a list of (what, raw line, what must
come back) — run through an `exchange` function that sends one raw line and returns every
response it provoked. Pointing `exchange` at `handle_line` in-process is the first use; in
section 12 the identical list runs over a real subprocess's pipes, where framing, stdout
discipline and "a notification gets NOTHING back" are tested for real rather than assumed.

In [ ]:
def _frame(msg: Any) -> bytes:
    """The harness's OWN framing, independent of your encode_message: a probe built with the
    code under test could not catch a bug in that code."""
    return (json.dumps(msg) + "\n").encode("utf-8")


def _expect_error(code: int, request_id: Any) -> Callable[[list[dict]], bool]:
    """Exactly one response: an error with `code`, carrying `request_id` (None = null or absent)."""
    return lambda got: (len(got) == 1 and got[0].get("id") == request_id
                        and got[0].get("error", {}).get("code") == code)


# (what it probes, the raw line sent, the check on every response it provoked, likely mistake)
_CONFORMANCE_CASES: tuple[tuple[str, bytes, Callable[[list[dict]], bool], str], ...] = (
    ("malformed JSON", b'{"jsonrpc": "2.0", "id": 1, "method": \n',
     _expect_error(PARSE_ERROR, None),
     "a line that is not JSON must get exactly one PARSE_ERROR answer with a null id"),
    ("unknown method", _frame({"jsonrpc": "2.0", "id": 7, "method": "tools/frobnicate"}),
     _expect_error(METHOD_NOT_FOUND, 7),
     "an unimplemented method must get METHOD_NOT_FOUND under the request's own id"),
    ("unsupported protocol version",
     _frame({"jsonrpc": "2.0", "id": 1, "method": "initialize",
             "params": {"protocolVersion": "1999-01-01", "capabilities": {},
                        "clientInfo": CLIENT_INFO}}),
     lambda got: (len(got) == 1 and "error" not in got[0]
                  and got[0].get("result", {}).get("protocolVersion") == LATEST_VERSION),
     "an unsupported version must get a successful result naming the server's latest version"),
    ("tools/call without a name",
     _frame({"jsonrpc": "2.0", "id": 2, "method": "tools/call",
             "params": {"arguments": {"key": "x"}}}),
     _expect_error(INVALID_PARAMS, 2),
     "a tools/call with no tool name is a protocol error: INVALID_PARAMS, same id"),
    ("a notification", _frame({"jsonrpc": "2.0", "method": "notifications/initialized"}),
     lambda got: got == [],
     "a notification must provoke NO response at all — not an error, not an empty line"),
    ("id: null", _frame({"jsonrpc": "2.0", "id": None, "method": "tools/list"}),
     _expect_error(INVALID_REQUEST, None),
     "id: null is an invalid request that must be answered, not a silent notification"),
    ("ping", _frame({"jsonrpc": "2.0", "id": "p", "method": "ping"}),
     lambda got: got == [{"jsonrpc": "2.0", "id": "p", "result": {}}],
     "ping must be answered promptly with an empty result"),
)


def run_conformance(exchange: Callable[[bytes], list[dict]]) -> int:
    """Send every case's raw line through `exchange` (which returns every response that line
    provoked, decoded) and check each. Stops at the first case that fails, naming it and the
    likely mistake; returns how many cases ran. Given to you."""
    for what, raw, ok, mistake in _CONFORMANCE_CASES:
        got = exchange(raw)
        assert ok(got), f"conformance case '{what}': got {got!r} — {mistake}"
    return len(_CONFORMANCE_CASES)


def _conformance_in_process() -> None:
    def exchange(raw: bytes) -> list[dict]:
        reg = make_registry(_DEMO_TASK.chain, _DEMO_TASK.target, 0.0, random.Random(0))
        out = handle_line(ServerState(registry=reg), raw)
        return [] if out is None else [json.loads(line) for line in out.splitlines()]

    cases = run_conformance(exchange)
    print(f"conformance harness, handle_line in-process: all {cases} cases behave as the spec "
          "requires")


_try("conformance harness", _conformance_in_process, needs=("exercise 7",))

## 10. Exercise 8 — `mcp_call()`

The client's half of a request/response round trip: build a JSON-RPC request with a fresh
id, send it, read back exactly one response, and check that the response's `id` actually
matches the id that was sent. The specification puts it as "Result responses MUST include
the same ID as the request they correspond to" (claims.yaml, mcp-response-id-matches-
request) — the rule `handle_line` follows from the other side. A client that skips this
check cannot tell its own answer apart from a stale one, or one meant for a different
in-flight request.

<details><summary>💡 Hint 1 — what to think about</summary>

A request is the same envelope `_protocol_error` builds, with a method instead of an error.
Which of its keys must be left out entirely, rather than sent as null, when there is nothing
to put in it? And how many times per call should `next_id()` be called?
</details>
<details><summary>💡 Hint 2 — the approach, in words</summary>

Take one fresh id, build the request dict around it (adding params only when you were given
some), frame it with exercise 1 and hand it to `send`. Read one line with `recv`, decode it
with exercise 2, and compare its id with the one you sent: on a mismatch raise the lesson's
client error with a message naming both ids; otherwise return the decoded response.
</details>

In [ ]:
def mcp_call(send: Callable[[bytes], None], recv: Callable[[], bytes],
            next_id: Callable[[], int], method: str, params: dict | None) -> dict:
    """Send one JSON-RPC request and return its decoded response, checking id agreement.

    `send` writes one already-framed line somewhere (a pipe, a queue); `recv` blocks for and
    returns exactly one framed line back. Raises `MCPClientError` if the response's "id" does
    not match the id this call sent — the request/response id-matching rule.

    Example:
        >>> sent = []
        >>> def send(b): sent.append(b)
        >>> def recv(): return encode_message({"jsonrpc": "2.0", "id": 1, "result": {}})
        >>> mcp_call(send, recv, lambda: 1, "ping", None)
        {'jsonrpc': '2.0', 'id': 1, 'result': {}}
        >>> sent
        [b'{"jsonrpc":"2.0","id":1,"method":"ping"}\\n']
    """
    # YOUR CODE HERE
    raise NotImplementedError


def _check_mcp_call() -> None:
    reg = make_registry(_DEMO_TASK.chain, _DEMO_TASK.target, 0.0, random.Random(0))
    state = ServerState(registry=reg)
    outbox: list[bytes] = []
    counter = [0]

    def next_id() -> int:
        counter[0] += 1
        return counter[0]

    def send(data: bytes) -> None:
        outbox.append(data)

    def recv_ok() -> bytes:
        line = outbox.pop(0)
        response = handle_line(state, line)
        assert response is not None, (
            f"the server gave no answer to {line!r} — a request mcp_call sends must carry an "
            "'id', or it is a notification and nothing comes back"
        )
        return response

    result = mcp_call(send, recv_ok, next_id, "tools/list", None)
    assert result["id"] == 1, f"the first call must use id 1 from next_id(), got {result['id']}"
    assert {t["name"] for t in result["result"]["tools"]} == {"lookup", "submit"}, f"got {result!r}"

    result2 = mcp_call(send, recv_ok, next_id, "tools/call",
                       {"name": "lookup", "arguments": {"key": _DEMO_TASK.start_key}})
    assert result2["id"] == 2, "the SECOND call must use a fresh id (2), not reuse the first"

    def recv_mismatched() -> bytes:
        outbox.pop(0)
        return encode_message({"jsonrpc": "2.0", "id": 999, "result": {}})

    try:
        mcp_call(send, recv_mismatched, next_id, "tools/list", None)
    except MCPClientError as exc:
        assert "999" in str(exc), f"the error should name the mismatched id, got: {exc}"
    else:
        raise AssertionError(
            "a response whose id does not match the request's id must raise MCPClientError — "
            "silently accepting any reply as the answer to any question is how a client "
            "hands a model the wrong tool result"
        )

    no_params = mcp_call(send, recv_ok, next_id, "tools/list", None)
    assert "error" not in no_params, "params=None must be omitted, not sent as null"


_try("exercise 8", _check_mcp_call)

## 11. What is out of scope, and why

This lesson implements what a stdio tool-calling client and server need from MCP revision
2025-11-25 — the handshake, `ping`, `tools/list` and `tools/call` — and nothing else
(claims.yaml, mcp-feature-families, for the lists below):

- **Resources and prompts** — the two other server feature families beside tools. Nothing
  here touches them.
- **Sampling, roots and elicitation** — the client features: requests a server sends INTO
  the client. This lesson's client never accepts an inbound request at all.
- **Streamable HTTP** — the second standard transport. The specification says clients
  "SHOULD support stdio whenever possible" (claims.yaml, mcp-clients-should-support-stdio).
- **Auth** — the specification says stdio implementations should NOT follow its HTTP
  authorization framework and should take credentials from the environment instead
  (claims.yaml, mcp-stdio-auth-from-environment); these tools need no credentials at all.
- **Cancellation, pagination, progress, logging** — each a small utility layered on the
  request/response shape this lesson already covers.
- **The 2026-07-28 per-request scheme** — see section 1: no handshake, a version on every
  request, `server/discover`, and extension negotiation.

## 12. The server loop, and a real subprocess proof

`_serve_stdio` is what serves stdio once a server process exists to run it: a thin, given
loop around `handle_line`, the function actually being tested. The demo below starts a real
server process at the other end of a pipe, and picks how at run time:

1. **`Path(__file__)` exists** (`python lesson.py`, this repository's own grader,
   `tools/execute.py`): relaunch THIS FILE as the subprocess, in server mode.
2. **No `__file__`** (any notebook cell: Colab, Kaggle, Binder, Codespaces, a local
   Jupyter): write a standalone server SCRIPT to a temporary file and launch that instead.
   `inspect.getsource` recovers your graded functions, and any helper, constant or import
   they use, from IPython's own record of each cell's source.

If some source cannot be recovered even that way, the identical server loop runs on a
background thread instead, wired to the client through a real `os.pipe()` pair: real bytes
and real framing, just not a second OS process. The demo's output says which one ran.

Either server process points `print()` at stderr before anything else runs (this file's
launcher line included) and writes MCP messages only through the real stdout handle, since
a stdio server's stdout may carry nothing else (claims.yaml, mcp-stdio-stdout-must-not-
carry-non-mcp-bytes). The in-thread server writes nothing but responses to its pipe.

In [ ]:
def _serve_stdio(state: ServerState, stdin_buf, stdout_buf) -> None:
    """Read framed lines from `stdin_buf` until EOF; write each handled response back
    immediately. Given to you — built entirely from handle_line, which is graded."""
    for raw in stdin_buf:
        if not raw.strip():
            continue
        response = handle_line(state, raw)
        if response is not None:
            stdout_buf.write(response)
            stdout_buf.flush()


def _reader_thread(pipe) -> "queue.Queue[bytes]":
    """A background thread draining `pipe` into a Queue, so reads can be given up on with a
    timeout — a plain blocking `readline()` cannot be. Given to you."""
    q: "queue.Queue[bytes]" = queue.Queue()

    def _drain() -> None:
        try:
            for line in iter(pipe.readline, b""):
                q.put(line)
        except (OSError, ValueError):   # the pipe was closed under this thread: EOF too
            pass
        finally:
            q.put(b"")
            with contextlib.suppress(OSError, ValueError):
                pipe.close()   # this thread owns the read end until it has seen EOF

    threading.Thread(target=_drain, daemon=True).start()
    return q


# The child's bootstrap when THIS file is the one being relaunched (the __file__ path only):
# send print() to stderr BEFORE the file's first line (the launcher cell) runs, then run the
# file as __main__ with --serve-mcp in its argv.
_SERVER_BOOTSTRAP = ("import runpy, sys; sys.stdout = sys.stderr; path = sys.argv[1]; "
                     "sys.argv = [path, '--serve-mcp']; runpy.run_path(path, run_name='__main__')")

# The notebook path's generated script opens the same way: print() goes to stderr before any
# other line of it runs, so nothing but the server loop's own writes ever reaches the client.
_SERVER_PRELUDE = ("from __future__ import annotations   # annotations stay text, never looked up\n"
                   "import sys\n"
                   "_REAL_STDOUT = sys.__stdout__\n"
                   "sys.stdout = sys.stderr\n")

# Then the given CLASSES a standalone server needs, and the one constant that is not a plain
# literal, as fixed text. inspect.getsource cannot recover these, ever, in exactly the
# situation the script exists for: for a class (unlike a function) it never even looks at
# linecache — inspect.getfile(cls) reads sys.modules[cls.__module__].__file__ and raises
# OSError outright when cls.__module__ == "__main__" has none, which every notebook kernel's
# __main__ always is (verified against this lesson's own classes in a real kernel). They are
# fixed, ungraded boilerplate, identical in every student's copy, so they are reproduced here.
_SERVER_GIVEN = '''
import random
from typing import Any, Callable, NamedTuple

_ARG_TYPES = {"string": str, "integer": int, "number": (int, float), "boolean": bool,
             "array": list, "object": dict}


class TransientToolError(Exception):
    """Raised by a tool for a failure worth retrying. Never raised for the caller's own mistake."""


class ToolResult(NamedTuple):
    ok: bool
    output: Any
    error: str | None
    transient: bool


class ToolRegistry(NamedTuple):
    tools: dict[str, Callable[..., Any]]
    schemas: dict[str, dict]


class ChainTask(NamedTuple):
    task_id: int
    start_key: str
    key_sequence: list[str]
    chain: dict[str, Any]
    target: int
    length: int


class ServerState:
    """Mutable per-connection state. `registry` is fixed for the connection's life."""

    def __init__(self, registry: ToolRegistry):
        self.registry = registry
        self.initialized = False
        self.negotiated_version: str | None = None


class JSONRPCParseError(Exception):
    """Raised by decode_message when a line is not valid JSON."""
'''

# What the generated script's own last lines call. Every other function, constant and import it
# needs (your graded functions, and any helper of your own they use) is found from these.
_SERVER_ROOTS = ("generate_chain_task", "make_registry", "_serve_stdio")
_GLOBAL_LOADS = {"LOAD_GLOBAL", "LOAD_NAME", "LOAD_FROM_DICT_OR_GLOBALS"}


def _globals_used(func: Callable, source: str) -> set[str]:
    """Every global name `func`'s code reads (nested functions and comprehensions included),
    plus the names its default values and decorators read when the script defines it."""
    names, codes = set(), [func.__code__]
    while codes:
        code = codes.pop()
        names |= {ins.argval for ins in dis.get_instructions(code) if ins.opname in _GLOBAL_LOADS}
        codes += [const for const in code.co_consts if inspect.iscode(const)]
    node = ast.parse(source).body[0]
    for expr in (*node.decorator_list, *node.args.defaults, *filter(None, node.args.kw_defaults)):
        names |= {n.id for n in ast.walk(expr) if isinstance(n, ast.Name)}
    return names


def _is_literal(value: Any) -> bool:
    """True when `repr(value)` is Python source that rebuilds an equal value."""
    try:
        return ast.literal_eval(repr(value)) == value
    except (ValueError, TypeError, SyntaxError, MemoryError, RecursionError):
        return False


def _build_server_script() -> str:
    """Assemble a standalone MCP server script: the prelude, the given classes, every function,
    constant and import that `_SERVER_ROOTS` reach — recovered live by name, so it is your code
    as it stands now, helpers included — and a main that serves stdio. Nothing in it imports
    back from this notebook.

    Raises OSError or TypeError (what `inspect.getsource` raises when a function's source line
    cannot be found, e.g. one typed at a bare REPL prompt, which linecache never sees),
    LookupError (a name bound to something a script cannot rebuild) or SyntaxError, so the
    caller can fall back instead of launching a script with a hole in it.
    """
    given: dict[str, Any] = {"sys": sys, "_REAL_STDOUT": None}
    exec(_SERVER_GIVEN, given)   # a throwaway namespace, only to learn which names it defines
    ns, carried, functions, todo, seen = globals(), [], [], list(_SERVER_ROOTS), set()
    while todo:
        name = todo.pop()
        if name in seen or name in given or name not in ns:
            continue   # done already, given as text, or a builtin (or unbound here as there)
        seen.add(name)
        obj = ns[name]
        home = getattr(obj, "__module__", None)
        if inspect.ismodule(obj):
            carried.append(f"import {obj.__name__} as {name}")
        elif home not in (None, __name__) and getattr(
                sys.modules.get(home), getattr(obj, "__qualname__", ""), None) is obj:
            carried.append(f"from {home} import {obj.__qualname__} as {name}")
        elif inspect.isfunction(obj) and obj.__name__ == name and obj.__closure__ is None:
            source = textwrap.dedent(inspect.getsource(obj))
            functions.append(source)
            todo += sorted(_globals_used(obj, source))
        elif _is_literal(obj):
            carried.append(f"{name} = {obj!r}")
        else:
            raise LookupError(f"{name!r} is a {type(obj).__name__}, which a generated script "
                              "cannot rebuild")
    main = ("if __name__ == '__main__':\n"
            f"    _task = generate_chain_task(length={_DEMO_TASK.length!r}, seed={SEED!r})\n"
            "    _registry = make_registry(_task.chain, _task.target, 0.0, random.Random(0))\n"
            "    _serve_stdio(ServerState(registry=_registry), sys.stdin.buffer,\n"
            "                 _REAL_STDOUT.buffer)\n")
    script = "\n".join(["# Generated by _build_server_script() in section 12: a real MCP server "
                        "built from your own code.", _SERVER_PRELUDE + _SERVER_GIVEN,
                        *sorted(carried), "", *functions, main])
    compile(script, "mcp_server.py", "exec")   # a SyntaxError surfaces here, not in the child
    return script


def _walk_chain_over_wire(send: Callable[[bytes], None], recv: Callable[[], bytes],
                          next_id: Callable[[], int], via: str) -> None:
    """The client half of the proof, shared by every transport section 12 can reach a real
    server through: the full handshake, the tools/list check, walking the demo chain task to
    completion purely over the wire, then the conformance harness again over the same wire.
    `via` names what is on the other end, for the printed proof."""

    def over_the_pipe(raw: bytes) -> list[dict]:
        # Send the probe, then a ping with a unique id. Every line that comes back before the
        # ping's answer was provoked by the probe — so "nothing" is observable, not assumed.
        sentinel = f"sentinel-{next_id()}"
        send(raw)
        send(_frame({"jsonrpc": "2.0", "id": sentinel, "method": "ping"}))
        provoked = []
        while True:
            msg = decode_message(recv())
            if isinstance(msg, dict) and msg.get("id") == sentinel:
                return provoked
            provoked.append(msg)

    t0 = time.perf_counter()
    init = mcp_call(send, recv, next_id, "initialize",
                    {"protocolVersion": PROTOCOL_VERSION, "capabilities": {},
                     "clientInfo": CLIENT_INFO})
    assert "result" in init, f"initialize failed: {init}"
    negotiated = init["result"]["protocolVersion"]
    accept_negotiated_version(frozenset(SUPPORTED_VERSIONS), negotiated)
    send(encode_message({"jsonrpc": "2.0", "method": "notifications/initialized"}))

    listed = mcp_call(send, recv, next_id, "tools/list", {})
    names = {t["name"] for t in listed["result"]["tools"]}
    assert names == set(TOOL_SCHEMAS), f"tools/list returned {names}"

    key = _DEMO_TASK.start_key
    calls = 0
    value = None
    while value is None and calls <= _DEMO_TASK.length:   # a chain never needs more
        resp = mcp_call(send, recv, next_id, "tools/call",
                        {"name": "lookup", "arguments": {"key": key}})
        calls += 1
        payload = json.loads(resp["result"]["content"][0]["text"])
        if "next_key" in payload:
            key = payload["next_key"]
        else:
            value = payload["value"]
    assert value is not None, f"no value after {calls} lookups on a length-" \
                              f"{_DEMO_TASK.length} chain"
    resp = mcp_call(send, recv, next_id, "tools/call",
                    {"name": "submit", "arguments": {"value": value}})
    calls += 1
    correct = json.loads(resp["result"]["content"][0]["text"])["correct"]
    elapsed = time.perf_counter() - t0
    assert correct, "the chain walk over MCP reached the wrong final value"
    print(f"[{via}] walked a length-{_DEMO_TASK.length} chain in {calls} MCP tool calls "
         f"({elapsed:.3f} s end to end); negotiated {negotiated}, server reported "
         f"correct={correct}")

    cases = run_conformance(over_the_pipe)
    print(f"[{via}] conformance harness over the real wire: all {cases} cases behave as the "
         "spec requires")


def _demo_via_subprocess(argv: list[str], via: str) -> None:
    """Run `_walk_chain_over_wire` against a real OS subprocess started with `argv`. Always
    terminated: stdin is closed to ask it to exit, then it is waited on with a timeout and
    killed outright if that timeout passes."""
    proc = subprocess.Popen(argv, stdin=subprocess.PIPE, stdout=subprocess.PIPE,
                            stderr=subprocess.PIPE)
    out_q = _reader_thread(proc.stdout)
    err_q = _reader_thread(proc.stderr)   # drained too, so a chatty child can never block
    counter = [0]

    def next_id() -> int:
        counter[0] += 1
        return counter[0]

    def send(data: bytes) -> None:
        proc.stdin.write(data)
        proc.stdin.flush()

    def stderr_tail() -> str:
        lines = []
        while not err_q.empty():
            lines.append(err_q.get_nowait())
        return b"".join(lines)[-1500:].decode("utf-8", "replace")

    def recv(timeout: float = 5.0) -> bytes:
        try:
            line = out_q.get(timeout=timeout)
        except queue.Empty:
            raise TimeoutError(f"the server subprocess did not answer within {timeout:.0f} s; "
                               f"its stderr ends: {stderr_tail()!r}") from None
        if not line:
            raise RuntimeError("the server subprocess closed its stdout unexpectedly; its "
                               f"stderr ends: {stderr_tail()!r}")
        return line

    try:
        _walk_chain_over_wire(send, recv, next_id, via=via)
    finally:
        try:
            proc.stdin.close()
        except Exception:
            pass
        try:
            proc.wait(timeout=5)
        except subprocess.TimeoutExpired:
            proc.kill()
            proc.wait()


def _demo_via_pipes(reason: str) -> None:
    """Run `_walk_chain_over_wire` against `_serve_stdio` on a background thread, connected to
    THIS process through a real `os.pipe()` pair rather than an in-memory queue — real file
    descriptors, real bytes, real newline framing, just not a second OS process. Used only when
    `_build_server_script` cannot build a script. Closing the client's write end sends the
    server loop real EOF, exactly as closing a subprocess's stdin does, and the thread is then
    joined with a timeout. Unlike a process, a thread stuck inside your code cannot be killed:
    it is a daemon thread, and it ends when the kernel does."""
    print(f"subprocess demo: falling back to an in-process server thread over a real "
         f"os.pipe() pair — {reason}")
    c2s_r, c2s_w = os.pipe()   # client writes here, server's stdin reads it
    s2c_r, s2c_w = os.pipe()   # server writes here, client's stdin reads it
    server_in = os.fdopen(c2s_r, "rb", buffering=0)
    server_out = os.fdopen(s2c_w, "wb", buffering=0)
    client_out = os.fdopen(c2s_w, "wb", buffering=0)
    client_in = os.fdopen(s2c_r, "rb", buffering=0)

    registry = make_registry(_DEMO_TASK.chain, _DEMO_TASK.target, 0.0, random.Random(0))
    state = ServerState(registry=registry)
    thread_errors: list[BaseException] = []

    def _serve() -> None:
        try:
            _serve_stdio(state, server_in, server_out)
        except BaseException as exc:   # never swallowed — surfaced through stderr_tail below
            thread_errors.append(exc)
        finally:
            try:
                server_out.close()
            except OSError:
                pass

    thread = threading.Thread(target=_serve, daemon=True)
    thread.start()
    out_q = _reader_thread(client_in)
    counter = [0]

    def next_id() -> int:
        counter[0] += 1
        return counter[0]

    def send(data: bytes) -> None:
        client_out.write(data)
        client_out.flush()

    def stderr_tail() -> str:
        return ("; ".join(f"{type(exc).__name__}: {exc}" for exc in thread_errors)
                or "(server thread reported nothing)")

    def recv(timeout: float = 5.0) -> bytes:
        try:
            line = out_q.get(timeout=timeout)
        except queue.Empty:
            raise TimeoutError(f"the server thread did not answer within {timeout:.0f} s; "
                               f"{stderr_tail()}") from None
        if not line:
            raise RuntimeError(f"the server thread closed its output pipe unexpectedly; "
                               f"{stderr_tail()}")
        return line

    try:
        _walk_chain_over_wire(send, recv, next_id, via="in-process thread over os.pipe()")
    finally:
        try:
            client_out.close()
        except OSError:
            pass
        thread.join(timeout=5)
        if not thread.is_alive():
            server_in.close()   # client_in is closed by its reader thread, at EOF


def _demo_subprocess_walk() -> None:
    if _SERVE_MODE:
        print("subprocess demo: skipped — this process IS the server subprocess.")
        return
    if _HAS_FILE:
        _demo_via_subprocess([sys.executable, "-c", _SERVER_BOOTSTRAP, str(_LESSON_FILE)],
                             via="real subprocess (relaunched lesson.py)")
        return
    try:
        script = _build_server_script()
    except (OSError, TypeError, LookupError, SyntaxError) as exc:
        _demo_via_pipes(reason=f"could not build a standalone server script "
                               f"({type(exc).__name__}: {exc})")
        return
    with tempfile.TemporaryDirectory(prefix="t07l02-mcp-server-") as tmp_dir:
        tmp_path = Path(tmp_dir) / "mcp_server.py"
        tmp_path.write_text(script, encoding="utf-8")
        _demo_via_subprocess([sys.executable, str(tmp_path)],
                             via="real subprocess (generated server script)")


_try("subprocess demo", _demo_subprocess_walk,
    needs=("exercise 1", "exercise 2", "exercise 4", "exercise 5", "exercise 6",
          "exercise 7", "exercise 8"))

## 13. Self-check

1. A client's `initialize` asks for a `protocolVersion` this server does not support. What
   does revision 2025-11-25's MUST rule require the server to send back?
   a) A JSON-RPC error, because negotiation has failed
   b) A successful result naming a different version the server does support, so the client
      can decide on its own side whether it can speak that version
   c) A successful result echoing the requested version, to keep the handshake going
   d) Nothing; the client is expected to retry with another version

2. A `tools/call` names a tool that exists, but with an argument that fails THAT TOOL's own
   schema. What must the server send back?
   a) A JSON-RPC error with the invalid-params code
   b) The same response as for an unknown tool name
   c) A successful result with `"isError": true`, its content saying what was wrong
   d) No response — a bad argument is treated like a notification

3. A message arrives with a real method name but no `"id"` key at all. What must happen?
   a) The server answers with `"id": null`
   b) The server rejects it as malformed
   c) The server answers, using the id of the previous request
   d) The server may act on it, and sends back NOTHING

4. What changed in the most current MCP revision (checked on this lesson's access date),
   compared with the 2025-11-25 revision this lesson implements?
   a) Nothing of substance; the handshake is identical
   b) stdio was removed as a transport
   c) JSON-RPC was replaced by a binary framing format
   d) The `initialize` handshake is gone: every request declares its own version, and every
      server must implement a `server/discover` RPC

5. `encode_message` could have used `json.dumps(msg, indent=2)`. What would that break?
   a) stdio's one-message-per-line framing, since the pretty-printed form puts real
      newlines between keys, which a reader takes as message boundaries
   b) Nothing — pretty-printed JSON parses the same as compact JSON
   c) Only readability in a log file, never correctness
   d) UTF-8 encoding would fail

<details><summary>Answers</summary>

Answers come with this lesson's worked solution when you enrol on Synapsa.
</details>

In [ ]:
_MARKS = {"passed": "✅", "failed": "❌", "not started": "⏳"}


def _progress_board() -> None:
    width = max(len(", ".join(funcs)) for funcs in _EXERCISES.values())
    print("progress board")
    for label, funcs in _EXERCISES.items():
        state = _STATUS.get(label, "not started")
        print(f"  {_MARKS[state]} {label:<12} {', '.join(funcs):<{width}}  {state}")
    done = sum(_STATUS.get(label) == "passed" for label in _EXERCISES)
    print(f"\n{done} of {len(_EXERCISES)} exercises complete")
    failing = [label for label in _EXERCISES if _STATUS.get(label) == "failed"]
    if failing:
        print("failing right now: " + ", ".join(failing) + ". Each one printed what went "
             "wrong in its own cell above, and every exercise has hints you can open.")
    elif done < len(_EXERCISES):
        print("work top to bottom: every exercise has hints you can open above its code.")

## What you built

`encode_message`/`decode_message` are the stdio wire itself; `check_request_shape` and
`dispatch` are the boundary that turns a malformed, unversioned or misdirected message into
the SPECIFIC response the specification requires, never a crash and never a guess.
`handle_line` composes them into the one function a real server loop needs, and `mcp_call`
is its mirror image on the client. None of that depended on a language model or on Python
at the other end of the pipe: `mcp_call` and the conformance harness only ever see framed
JSON lines, so they would speak to any server that follows revision 2025-11-25, written in
any language.

In [ ]:
# Your progress board. Every check is re-run here, quietly, against your code as it stands
# now, so the board is current even if you edited an exercise and did not re-run its cell.
if __name__ == "__main__":
    if _SERVE_MODE:
        _serve_registry = make_registry(_DEMO_TASK.chain, _DEMO_TASK.target, 0.0, random.Random(0))
        _serve_state = ServerState(registry=_serve_registry)
        # sys.stdout stays pointed at stderr while serving: a print() inside your own code
        # must never land between two responses on the stream the client reads.
        _serve_stdio(_serve_state, sys.stdin.buffer, _REAL_STDOUT.buffer)
        raise SystemExit(0)

    _ALL_CHECKS = (
        ("exercise 1", _check_encode_message, ()),
        ("exercise 2", _check_decode_message, ()),
        ("exercise 3", _check_check_request_shape, ()),
        ("exercise 4", _check_negotiate_version, ()),
        ("exercise 5", _check_accept_negotiated_version, ()),
        ("exercise 6", _check_dispatch, ()),
        ("exercise 7", _check_handle_line, ()),
        ("conformance harness", _conformance_in_process, ("exercise 7",)),
        ("exercise 8", _check_mcp_call, ()),
        ("subprocess demo", _demo_subprocess_walk,
         ("exercise 1", "exercise 2", "exercise 4", "exercise 5", "exercise 6", "exercise 7",
          "exercise 8")),
    )
    with contextlib.redirect_stdout(io.StringIO()):
        for _name, _check, _needs in _ALL_CHECKS:
            _try(_name, _check, needs=_needs)
    _progress_board()
    print(f"\nnotebook wall time so far: {time.perf_counter() - _LESSON_T0:.1f}s")
    if _FAILED_CHECKS and "ipykernel" not in sys.modules:
        raise SystemExit("checks failed: " + ", ".join(dict.fromkeys(_FAILED_CHECKS)))

<!-- COMMONS NOTICE v1 · generated by tools/notebooks.py · do not edit by hand -->
---
**Synapsa Commons** · © 2026 RealAI · licensed under [CC BY-NC-SA
4.0](https://creativecommons.org/licenses/by-nc-sa/4.0/)

**You may** use this lesson to learn and to teach, and copy, fork, share and adapt it.

**You must** credit "Synapsa Commons by RealAI" with a link to
https://github.com/TarrySingh/Artificial-Intelligence-Deep-Learning-Machine-Learning-Tutorials,
say what you changed, and share anything you adapt under this same licence.

**You may not** use it, or anything adapted from it, in a way primarily intended for
commercial advantage or payment: for example selling it, charging for a course, bootcamp or
training built on it, or packaging it into a paid product or service. For a commercial
licence, contact [RealAI](https://www.realai.eu/contact).

Third-party material in this lesson keeps its own licence, named in `assets/SOURCE.md` or
`claims.yaml`. The Synapsa name and logo belong to RealAI and are not licensed. This summary
is not the licence: the [legal
code](https://creativecommons.org/licenses/by-nc-sa/4.0/legalcode) governs.